# 06. Single point

- Teaches the calculation every later module is built on: hand a structure to a machine-learning interatomic potential and ask it, once, for the energy and the forces of that structure exactly as it stands
- A machine-learning interatomic potential (MLIP) is a model trained on quantum-mechanical calculations that returns the energy and the forces of a structure in a fraction of a second, where the calculation it was trained on takes hours
- A single point is one such call at one fixed geometry: nothing is moved, nothing is optimized, and the answer describes the structure you gave it and no other
- The model is MACE-MP-0, reached through `mliprun`, the group's tool, and the structures are the clean Pt(1 1 1) slab of module 04 and the four O sites of module 05
    - A head is one output of a model, trained on one dataset: ask that head for an energy and you get an energy on that dataset's scale
    - MACE-MP-0 has a single head, so naming the model names the head as well; other models used in the group, UMA above all, carry several heads in one file, fitted separately and with independent energy zeros, and there the head has to be named next to the model
    - Head and model together are what canon `C1` pins, and every energy in this notebook comes from the one head of `mace`
- Before you start: modules 00 to 05 finished, the `mlip-mace` environment active, and Jupyter Lab running
    - The kernel shown at the top right of this notebook is "Python 3 (ipykernel)", the Python of that environment
    - The model weights are already on your computer: step 10 of module 00 loaded the model once, which downloaded them, about 45 MB, into `~/.cache/mace/`; nothing is downloaded here, and a computer where that step was never run downloads them at the first call of this notebook instead
- After this you can attach an MLIP to any structure, read its energy and its forces, build one model and reuse it for many structures, assemble an energy difference from terms that all come from the same model, and read the forces to see how far a placed adsorbate is from a minimum
- The last section runs the same single point from the terminal, with the command `mlip singlepoint run`, and reads the record of the run that it writes
- Every energy printed in this notebook is a single point on an unrelaxed geometry, and none of them is a result to quote anywhere
    - The adsorbate heights come from module 05's table of starting guesses, and the forces of the section on forces measure how far from a minimum those guesses are
    - Module 07 relaxes them; the energy differences of the section on references below are where that relaxation starts, and nothing more

### Before you run this for a project

- The values in this notebook are teaching values, chosen so it runs on a laptop in minutes
- In a project, the items below are decided by your supervisor and written in the project's `CANON.md`; they are never taken from a tool default
    - `C1` MLIP model and head: this notebook uses `mace` (MACE-MP-0); the project may use another
    - `C3` one head per energy formula: every term of an energy difference comes from one model and one head, so the choice recorded for `C1` is forced on every term of every formula
- If a value is not recorded, ask before running; the tool will not stop you

## 0. The setup cell

- Run this cell first, every time you open the notebook
- Every structure is built in memory by the helper, as module 05's section on saving the structures said the calculation modules would
    - The one exception is the last section: a terminal command reads its structure from a file, so that section writes one, and the command's output, into `work/single_point_fcc/`
- `sys.path` is the list of folders Python searches when it imports something
    - `sys.path.insert(0, ...)` puts the repository's `scripts` folder at the front of that list, and without it `from onboarding_structures import ...` fails, because `scripts` is not next to this notebook
    - `Path("../../scripts").resolve()` turns the relative path into a full path, and `str(...)` turns that path into text, which is what `sys.path` holds
    - The cell prints the path as it is written, `../../scripts`; the full path it resolves to depends on where you keep this repository, so it is different on every computer
    - `sys`, `pathlib`, `io`, `json`, `logging`, `subprocess`, `time`, `warnings` and `contextlib` all come with Python
- Three functions come out of `mliprun`, and they are the whole Python interface this course uses to an MLIP; its terminal commands are a separate thing, and the last section of this notebook comes to them
    - `setup_calculator(atoms, mlip=..., device=...)` builds a model and attaches it to one structure, used in the section on attaching a calculator
    - `build_calculator(mlip=..., device=...)` builds the same model and gives it back without attaching it, used in the section on reusing one calculator
    - `calc_fmax(forces)` reduces a whole forces array to one number, used from the section on energy and forces onward
- `numpy` is imported under the short name `np` and `pandas` under `pd`, as in modules 01 to 05
    - NumPy holds the forces of this notebook and does the arithmetic on them; pandas prints its tables
- `LOAD_LOG = io.StringIO()` makes a text buffer in memory, and it is the only piece of output handling this cell sets up
    - The two cells that build a model write the loader's own printed lines into that buffer instead of into the notebook, and the section on attaching a calculator says what those lines contain and how to read them
    - Loading a model also raises the warnings that step 10 of module 00 told you to ignore; they are silenced inside the two model-load blocks and nowhere else, as module 03's section on querying the database did with its own call, so nothing that a later cell reports is hidden
    - Silencing is never set up once for a whole notebook here: modules 07 to 10 copy this cell, and a relaxation or an MD run whose warnings were switched off at the top would be a worse thing than a noisy output
- `write` comes from `ase.io`, as in module 05, and is used only in the last section
- `SITES` is the four sites of module 05, in the order every table below uses them
- `MLIP = Path(sys.executable).parent / "mlip"` is the `mlip` command of this environment, found next to the Python that runs this notebook; in the terminal you type `mlip` on its own, with the `mlip-mace` environment active

In [1]:
import io
import json
import logging
import subprocess
import sys
import time
import warnings
from contextlib import redirect_stdout
from pathlib import Path

import numpy as np
import pandas as pd
from ase.io import write

sys.path.insert(0, str(Path("../../scripts").resolve()))
from onboarding_structures import pt111_slab, pt111_with_adsorbate, molecule_in_box

from mliprun.cli.utils import build_calculator, setup_calculator
from mliprun.core.utils import calc_fmax

LOAD_LOG = io.StringIO()
SITES = ("ontop", "bridge", "fcc", "hcp")
MLIP = Path(sys.executable).parent / "mlip"

print("Folder the structures helper was imported from")
print(Path("../../scripts"))
print("Sites this notebook puts an O atom on")
print(list(SITES))

slab = pt111_slab()

print()
print("Atoms in the clean Pt(1 1 1) slab")
print(len(slab))
print("Fixed atoms in it")
print(len(slab.constraints[0].get_indices()))
print("Atoms free to move")
print(len(slab) - len(slab.constraints[0].get_indices()))

Folder the structures helper was imported from
../../scripts
Sites this notebook puts an O atom on
['ontop', 'bridge', 'fcc', 'hcp']

Atoms in the clean Pt(1 1 1) slab
27
Fixed atoms in it
9
Atoms free to move
18


## 1. Attach a calculator

- A calculator is the object that answers two questions about a structure: what is its energy, and what is the force on each of its atoms
    - In ASE a structure keeps its calculator in `atoms.calc`, and nothing is calculated when the calculator is attached; the work happens the first time something asks for a number
    - Every calculator answers the same way, whether an MLIP or VASP sits behind it, which is why the two calls of the next section work whichever one is attached
- `setup_calculator(atoms, mlip="mace", device="cpu")` builds the model and attaches it in one call
    - It changes the structure it is given, the way `add_adsorbate` did in module 05, and also returns it
    - Building the model is the expensive part: the weights are read from disk into memory, and the printed number of seconds below is what that costs
- `mlip="mace"` is MACE-MP-0 medium, a foundation model trained on the Materials Project's PBE calculations of inorganic crystals
    - Foundation model means one model fitted to a large dataset covering most of the periodic table, meant to be used as it comes rather than fitted to your own system
    - The training data is bulk crystals, so a slab, an adsorbate and a free molecule are all further from what the model saw than a bulk oxide is; the section on references comes back to this
- The model name is written out in every call of this course, and never left to the tool
    - Leaving `--mlip` out of a terminal command is what lets mliprun choose the model for you, and it chooses in a fixed order of preference, taking the first of UMA, MACE, SevenNet, CHGNet that is installed
    - So the same command, run where a different set of models is installed, resolves to a different model and so to a different level of theory, while looking like the same command in your script, your notes and your job file
    - These Python calls take a model name only: passing `auto` as the model returns no calculator, and the error appears later as `RuntimeError: Atoms object has no calculator`; the `device="auto"` further down is a different argument and does work
    - The tool's own reasoning makes the sharper case: in an environment carrying only SevenNet, `auto` lands on a model with several tasks and then stops, because the task has no default and canon `C1` makes it an explicit decision; a stop can be recovered from, a wrong energy zero cannot
    - Canon `C1` is the decision that a project pins one model tag and one head, so `mlip="mace"` is spelled out here even though only one model is installed
- `device="cpu"` runs the model on the processor; `"cuda"` runs it on an NVIDIA GPU, and `"auto"` picks the GPU when there is one
    - This course names `"cpu"`, so the notebook does the same thing on a laptop and on a cluster login node
- Loading the model prints and warns, and the lines around the call hold that back for the length of the call and no longer
    - `with redirect_stdout(LOAD_LOG):` sends what the loader prints into the buffer of the setup cell instead of into the notebook
    - `with warnings.catch_warnings():` makes every warning setting inside the block temporary and `warnings.simplefilter("ignore")` hides the warnings raised there, and nowhere else in the notebook, which is module 03's pattern in its section on querying the database
    - The `WARNING:root:Default dtype float32 ...` line arrives through Python's logging system instead, which the filter above does not reach, so the root logger has to be turned down and turned back up by hand: `previous_level = root_logger.level` saves the level as it was before anything was changed, `root_logger.setLevel(logging.ERROR)` turns it down for the load, and the last line puts the saved level back
    - `try:` with `finally:` is what makes that last line certain: whatever happens inside the `try`, including an error that stops the cell, the `finally` block still runs, so a failed model load cannot leave the logger turned down for the rest of the session
    - The two `with` blocks are written on one line, separated by a comma, and both end together; unlike the logger, each of them puts back what it changed on its own
- What is being held back, and how to see it for yourself
    - The printed lines are the ones starting with `Using` that step 10 of module 00 showed: one names the weights file inside `~/.cache/mace/`, which is a different full path on every computer, and one names the numerical precision
    - The warnings are the `FutureWarning` and `UserWarning` lines of that same step, which PyTorch and MACE raise on every model load and which say nothing about your structure
    - To read all of it, call `build_calculator(mlip="mace", device="cpu")` in a cell of your own without these lines around it
    - The precision line says the model runs in single precision, which means each number is stored in 32 bits instead of the usual 64, and the message itself calls that faster and less accurate; so a small difference between two energies below is not automatically physical
- `time.perf_counter()` is a clock in seconds, used twice around a piece of work to measure how long it took
- `type(slab.calc).__name__` is the name of the class of the attached calculator, which says which package is actually answering

In [2]:
root_logger = logging.getLogger()
previous_level = root_logger.level

t_start = time.perf_counter()
try:
    with warnings.catch_warnings(), redirect_stdout(LOAD_LOG):
        warnings.simplefilter("ignore")
        root_logger.setLevel(logging.ERROR)
        setup_calculator(slab, mlip="mace", device="cpu")
finally:
    root_logger.setLevel(previous_level)
load_seconds = time.perf_counter() - t_start

print("Calculator now attached to the clean slab")
print(type(slab.calc).__name__)
print("Seconds the model took to build")
print(round(load_seconds, 1))

print()
print("Lines the loader printed, held in the buffer instead of the notebook")
print(len(LOAD_LOG.getvalue().strip().splitlines()))

Calculator now attached to the clean slab
MACECalculator
Seconds the model took to build
3.3

Lines the loader printed, held in the buffer instead of the notebook
3


## 2. Energy and forces

- `atoms.get_potential_energy()` is the total energy of the whole structure in eV, as this model defines it
    - It is not an adsorption energy, a formation energy or a binding energy: it is one number for 27 Pt atoms together, and on its own it says nothing
    - Only a difference between two such numbers from the same model means anything, and the section on references builds one
- `atoms.get_forces()` is the force on every atom, as an array with one row per atom and three columns, x, y and z, in eV/Å
    - A force is minus the slope of the energy against that atom's position, so its direction is the way the atom would move if it were let go, and its length is how hard it is being pushed
    - Zero force on every atom means the structure sits at a stationary point of this model's energy, a geometry where the energy does not change for any small move of any atom, which a minimum is one kind of; a large force means it is not at one
- `calc_fmax(forces)` from `mliprun.core.utils` reduces that array to one number: the length of the longest force vector in it, in eV/Å
    - It takes the forces array and not the structure, so it knows nothing about which atoms are fixed
    - It does not need to: `atoms.get_forces()` writes exact zeros into the rows of the atoms held by `FixAtoms`, so the maximum over all rows is already the maximum over the free atoms
    - Remove the constraint and the same call starts reporting the bottom layer as well, which is what the two numbers printed below for the structure with nothing fixed show
- The clean slab is not a relaxed surface either, and its forces say so
    - The slab was built in module 04 at ASE's reference lattice constant for Pt, so the spacing between its layers is that of ideal bulk and not the spacing this model prefers at a surface
    - The force on every atom of the top layer is along z alone, because a flat close-packed surface offers no in-plane direction for an atom to lower its energy in
    - The slab is centrosymmetric, meaning there is a point in it such that moving every atom straight through that point to the same distance on the other side gives the structure back; here that point sits on an atom of the middle layer
    - That is what makes the three layers come out as they do: the middle layer feels no net force, and the bottom layer feels the same force as the top one with the sign reversed, so the whole slab is pulled in neither direction
    - The symmetry is that inversion centre and not a reflection (a mirror, as in module 05's section on rotating an adsorbate): the layers of a close-packed metal are stacked in the repeating order A, B, C, so turning the slab over about its middle layer does not give the same stacking back
- `forces.shape` is the size of the array in each direction, printed as atoms by directions
- `np.linalg.norm(forces, axis=1)` is the length of each row of the array, so one force magnitude per atom, in eV/Å
- `(magnitudes == 0).sum()` counts how many of those magnitudes are exactly zero, and `int(...)` makes it a plain number
- `magnitudes[fixed].max()` is the largest magnitude among the rows listed in `fixed`, which is the largest force on an atom that is held in place
- `in_layer = free_slab.get_tags() == layer` is `True` for the atoms of one layer and `False` for the rest, the tags of module 04, and `f = free_forces[in_layer]` then keeps only those atoms' rows, so the last table is built one layer at a time
    - `int(in_layer.sum())` counts the atoms of the layer, 9 in each
    - `free_slab.positions[in_layer, 2].mean()` averages the z of those atoms, which is the height of the layer and puts the three rows of the table in order
    - `f[:, 2].mean()` averages the third column of the kept rows, the z component of the force, and `f[:, 2].max() - f[:, 2].min()` is the spread of that column within the layer
    - `np.abs(f[:, :2]).max()` is the largest force in x or y anywhere in the layer, which is how the claim of no in-plane force is checked rather than assumed
- `pd.DataFrame(rows).to_string(index=False)` prints a list of dictionaries as an aligned table without row numbers, as in modules 01 to 05
- `slab.copy()` makes a separate structure, as in modules 02 and 05, so the constraint of `slab` itself is left alone
    - `free_slab.set_constraint()` with nothing in the brackets removes every constraint from that copy
    - `free_slab.calc = slab.calc` hands the copy the model that is already in memory, which is the pattern the next section is about
- In the printed output:
    - The energy is a large negative number, because it counts every bond in the slab; its size carries no information about the surface
    - 9 rows of the forces array are exactly zero, one per fixed atom of the bottom layer
    - The largest force on a fixed atom is 0.0 eV/Å through `atoms.get_forces()` and 0.392 eV/Å once the constraint is removed, and the difference is the constraint and not the model
- In the table of layers, built from the copy with nothing fixed, so that the bottom layer is visible:
    - Both outer layers are pulled outward, away from the middle of the slab, each at 0.392 eV/Å per atom
    - Their z components carry opposite signs, `+0.392` on top and `-0.392` at the bottom, only because outward is `+z` at the top face and `-z` at the bottom one; the two faces behave the same way, as the inversion centre requires
    - The middle layer, the one the inversion centre sits in, is at zero, printed as `-0.000` because the rounding kept the sign of a value just below zero
    - The spread within each layer and the largest in-plane force are both 0.0 to the three decimals printed, so every atom of a layer feels the same force and none of it lies in the surface plane

In [3]:
energy = slab.get_potential_energy()
forces = slab.get_forces()

print("Total energy of the clean slab (eV)")
print(round(float(energy), 3))
print("Shape of the forces array, atoms by directions")
print(forces.shape)
print("Largest force on any atom (eV/Å)")
print(round(calc_fmax(forces), 3))

magnitudes = np.linalg.norm(forces, axis=1)
fixed = slab.constraints[0].get_indices()

print()
print("Rows of the forces array that are exactly zero")
print(int((magnitudes == 0).sum()))
print("Fixed atoms in the slab")
print(len(fixed))
print("Largest force on a fixed atom, as get_forces reports it (eV/Å)")
print(round(float(magnitudes[fixed].max()), 3))

free_slab = slab.copy()
free_slab.set_constraint()
free_slab.calc = slab.calc
free_forces = free_slab.get_forces()

print()
print("Largest force on those same atoms with nothing fixed (eV/Å)")
print(round(float(np.linalg.norm(free_forces[fixed], axis=1).max()), 3))
print("Largest force on any atom with nothing fixed (eV/Å)")
print(round(calc_fmax(free_forces), 3))

layer_rows = []
for layer in (1, 2, 3):
    in_layer = free_slab.get_tags() == layer
    f = free_forces[in_layer]
    layer_rows.append({"layer (tag)": layer,
                       "atoms": int(in_layer.sum()),
                       "mean z (Å)": round(float(free_slab.positions[in_layer, 2].mean()), 3),
                       "force along z (eV/Å)": round(float(f[:, 2].mean()), 3),
                       "spread in z within the layer (eV/Å)": round(float(f[:, 2].max() - f[:, 2].min()), 3),
                       "largest force in x or y (eV/Å)": round(float(np.abs(f[:, :2]).max()), 3)})

print()
print("Forces per layer of the clean slab, nothing fixed, top layer first")
print(pd.DataFrame(layer_rows).to_string(index=False))

Total energy of the clean slab (eV)
-153.478
Shape of the forces array, atoms by directions
(27, 3)
Largest force on any atom (eV/Å)
0.392

Rows of the forces array that are exactly zero
9
Fixed atoms in the slab
9
Largest force on a fixed atom, as get_forces reports it (eV/Å)
0.0

Largest force on those same atoms with nothing fixed (eV/Å)
0.392
Largest force on any atom with nothing fixed (eV/Å)
0.392

Forces per layer of the clean slab, nothing fixed, top layer first
 layer (tag)  atoms  mean z (Å)  force along z (eV/Å)  spread in z within the layer (eV/Å)  largest force in x or y (eV/Å)
           1      9      14.526                 0.392                                  0.0                             0.0
           2      9      12.263                -0.000                                  0.0                             0.0
           3      9      10.000                -0.392                                  0.0                             0.0


## 3. Reuse one calculator for many structures

- Building the model is the slow step and calling it is the fast one, so a model is built once and used for every structure of a study
    - `build_calculator(mlip="mace", device="cpu")` builds the same model as the section on attaching a calculator and returns it without attaching it to anything
    - The same `try` and `finally` and the same two `with` blocks as that section wrap this build, for the same reason and for no longer than the call itself
    - `atoms.calc = calc` attaches that one object to a structure, and the same object can be attached to as many structures as you like
    - `setup_calculator` builds a fresh model on every call, so calling it inside a loop over 40 structures reads the weights from disk 40 times; one model build costs more than all four single points below put together, so in such a loop the build, and not the calculation, is what every turn spends its time on
- The four structures are built by `pt111_with_adsorbate("O", site)`, the one-line helper of module 05
    - 28 atoms each: the 27 Pt of the slab plus the O, which is the last atom
    - The same 9 Pt atoms of the bottom layer stay fixed, so 19 atoms are free in every one of them
    - The adsorbate height is module 05's starting guess for that site, 1.85 Å ontop, 1.45 Å bridge and 1.30 Å in both hollows
- Using one model for every structure of a table is also what canon `C3` needs, and the next section is where that matters
- In the table:
    - The total energies of the four structures can be compared with each other, because all four hold the same 28 atoms and came from the same model
    - None of them can be compared with the clean slab's energy of the previous section, which counts 27 atoms; that comparison is the next section's subject and needs a reference for the extra O
    - `fcc` is the lowest of the four by 0.064 eV over `hcp`, with `bridge` 0.331 eV above `fcc` and `ontop` 1.084 eV above it
    - Every fmax is between 0.9 and 2.1 eV/Å, which is very far from a relaxed structure; the section on forces is about that

In [4]:
try:
    with warnings.catch_warnings(), redirect_stdout(LOAD_LOG):
        warnings.simplefilter("ignore")
        root_logger.setLevel(logging.ERROR)
        calc = build_calculator(mlip="mace", device="cpu")
finally:
    root_logger.setLevel(previous_level)

structures = {}
energies = {}
forces_by_site = {}

t_start = time.perf_counter()
for site in SITES:
    atoms = pt111_with_adsorbate("O", site)
    atoms.calc = calc
    structures[site] = atoms
    energies[site] = float(atoms.get_potential_energy())
    forces_by_site[site] = atoms.get_forces()
loop_seconds = time.perf_counter() - t_start

rows = []
for site in SITES:
    atoms = structures[site]
    rows.append({"site": site,
                 "atoms": len(atoms),
                 "free atoms": len(atoms) - len(atoms.constraints[0].get_indices()),
                 "total energy (eV)": round(energies[site], 3),
                 "fmax (eV/Å)": round(calc_fmax(forces_by_site[site]), 3)})

print("One O atom on each of the four sites of Pt(1 1 1), one single point each")
print(pd.DataFrame(rows).to_string(index=False))

print()
print("Seconds for the four single points, model already in memory")
print(round(loop_seconds, 2))

One O atom on each of the four sites of Pt(1 1 1), one single point each
  site  atoms  free atoms  total energy (eV)  fmax (eV/Å)
 ontop     28          19           -157.113        0.995
bridge     28          19           -157.866        1.514
   fcc     28          19           -158.197        1.783
   hcp     28          19           -158.133        2.023

Seconds for the four single points, model already in memory
1.15


## 4. Energy differences and references

- A total energy becomes a number about chemistry only inside a difference, and for an adsorbate the usual difference is `E_ads = E(slab+O) − E(slab) − ½ E(O2)`
    - `E(slab+O)` is the single point of the previous section, 28 atoms
    - `E(slab)` is the clean slab, 27 atoms
    - `½ E(O2)` is the reference for the one O atom that the two structures differ by: half the energy of an O2 molecule, because an O2 molecule holds two O atoms
    - Written this way, a negative `E_ads` means the model puts the O on the surface at a lower energy than half an O2 molecule, and a positive one means the opposite
- Every term must come from the same model and the same head, and this cell is built to show it rather than say it
    - All three terms are calculated with the one calculator object of the previous section: the four structures, the clean slab rebuilt here, and the O2 in its box
    - Canon `C3` is the rule: two models, or two heads of one model, have independent energy zeros, because each one was fitted separately and the zero of an energy is not part of what the fit constrains
    - Mix them and the difference contains the gap between two zeros, which is not a physical quantity and is far larger than any adsorption energy; the result looks like a number and is arithmetic on two different scales
    - The clean slab is deliberately recalculated here with this section's calculator, and the printed difference against the first section's number is zero to six decimal places: two separate builds of one model and one head give the same energy, so what `C3` is about is the model and the head and not which object computed the number
- `½ E(O2)` is one convention among several, and it is the weakest term in the expression
    - MACE-MP-0 was fitted to Materials Project calculations of crystals, so a single O2 molecule alone in a box is further from its training data than either slab is, and this term therefore carries more of the error than the two slab terms
    - The oxygen-evolution literature often references water instead, `H2O` and `H2` at a stated potential, which avoids the O2 molecule entirely
    - Which reference a project uses is part of how its energies are defined, so it is recorded and never swapped between two numbers that are compared
- `molecule_in_box("O2")` is the helper of module 05: ASE's tabulated O2 geometry, centred in a 12 Å cubic box with periodic boundaries, so the molecule is far from its own copies
    - Its bond length is a tabulated value and not this model's own, which is why the force printed in the box is not zero: let go, this model would change that bond length
- The `E_ads` column below is labelled unrelaxed for a reason: nothing in this notebook has been relaxed
    - Every term is a single point, two of them on geometries built from a table of starting heights and the third on a tabulated bond length
    - These numbers are where module 07's relaxation starts; its section on adsorption energies from relaxed structures is where an `E_ads` that can be reported comes from
    - Do not quote a number from this table, and do not compare it with a published adsorption energy
- Oxygen does bind to Pt(1 1 1), and the positive sign in this table is a statement about these geometries and not about the chemistry
    - Relax these structures and the clean slab with this same model and this same ½ E(O2) reference and every one of the four numbers falls, because both slabs are being held at geometries the model did not choose and the energy a relaxation would release has not been released yet
    - At the two hollow sites it falls far enough to change the sign, which is the binding this table is hiding
    - It does not change the sign everywhere: `ontop`, the most weakly bound of the four and the highest in the table at 1.209 eV, stays positive even after it is relaxed, which says how weakly the O sits on that one site and not that O fails to bind to platinum
    - Not every site relaxes to a minimum at all: a site that is a saddle rather than a minimum lets the adsorbate slide towards a neighbouring site instead of settling
    - Module 07 checks two things after each relaxation
        - Whether the run converged, canon `K1`: a calculation is reported as converged or not converged, and a non-converged energy is never mixed with converged ones; this label does not show whether the adsorbate slid
        - Where the adsorbate ended up, canon `A1`: this is the check that shows a slide, because the site name says where it was put and not where it ended up
    - Module 07 relaxes all four and prints the numbers, so that is where you see which of them end up negative
    - Read this table as the starting point of a relaxation, never as the answer to whether something binds; that question needs the relaxed numbers
- In the table:
    - All four `E_ads` come out positive, so this model, on these unrelaxed geometries, puts the O atom on the surface above half an O2 molecule in energy
    - The ranking is `fcc` lowest, then `hcp` 0.063 eV above it, then `bridge`, then `ontop` 1.084 eV above `fcc`, which is the ranking of the previous section's total energies and has to be, because every row differs from every other by the same two reference terms
    - That `fcc` to `hcp` separation is the same one the previous section read as 0.064 eV; each is the difference of two numbers already rounded to three decimals, in one case from the total-energy column and in the other from the `E_ads` column, and 0.001 eV is what that rounding costs
    - `fcc` and `hcp` are separated by less than 0.1 eV, and a relaxation moves each structure by much more than that, so this ranking of the two hollows is a hypothesis to test in module 07 and not an answer

In [5]:
clean = pt111_slab()
clean.calc = calc
e_clean = float(clean.get_potential_energy())

o2 = molecule_in_box("O2")
o2.calc = calc
e_o2 = float(o2.get_potential_energy())
e_reference = 0.5 * e_o2

print("Clean slab energy from this section's calculator (eV)")
print(round(e_clean, 3))
print("Difference from the energy the first section printed (eV)")
print(round(e_clean - float(energy), 6))

print()
print("Total energy of the O2 molecule in its box (eV)")
print(round(e_o2, 3))
print("Reference for one O atom, half of that (eV)")
print(round(e_reference, 3))
print("O to O distance in the box (Å)")
print(round(float(o2.get_distance(0, 1)), 3))
print("Largest force in the box (eV/Å)")
print(round(calc_fmax(o2.get_forces()), 3))

rows = []
for site in SITES:
    rows.append({"site": site,
                 "E(slab+O) (eV)": round(energies[site], 3),
                 "E(slab) (eV)": round(e_clean, 3),
                 "half E(O2) (eV)": round(e_reference, 3),
                 "E_ads, unrelaxed (eV)": round(energies[site] - e_clean - e_reference, 3)})

print()
print("E_ads = E(slab+O) − E(slab) − ½ E(O2), every term from one calculator, nothing relaxed")
print(pd.DataFrame(rows).to_string(index=False))

Clean slab energy from this section's calculator (eV)
-153.478
Difference from the energy the first section printed (eV)
0.0

Total energy of the O2 molecule in its box (eV)
-9.687
Reference for one O atom, half of that (eV)
-4.843
O to O distance in the box (Å)
1.246
Largest force in the box (eV/Å)
0.489

E_ads = E(slab+O) − E(slab) − ½ E(O2), every term from one calculator, nothing relaxed
  site  E(slab+O) (eV)  E(slab) (eV)  half E(O2) (eV)  E_ads, unrelaxed (eV)
 ontop        -157.113      -153.478           -4.843                  1.209
bridge        -157.866      -153.478           -4.843                  0.456
   fcc        -158.197      -153.478           -4.843                  0.125
   hcp        -158.133      -153.478           -4.843                  0.188


## 5. Forces tell you the structure is not relaxed

- The forces of a single point are how you know, without running anything else, that a geometry is only a starting guess
    - A relaxation stops when the largest force on a free atom falls below a chosen value; 0.05 eV/Å is the value module 07 uses
    - How tight that value is, is a per-project decision, canon `C2`, and module 07 is where it is introduced; nothing in this notebook relaxes, so 0.05 eV/Å is used here only as a yardstick to read the forces against
- Every one of the four placed structures has an fmax between about 20 and 40 times that yardstick, and the clean slab under them about 8 times it
    - These are starting heights from a table, so a large force is the expected outcome and not a mistake in the placement
    - The atom carrying the largest force is the O itself at `bridge`, `fcc` and `hcp`, so its own position is the furthest from where this model would put it
    - At `ontop` the largest force sits on a Pt atom instead, and the force on the O there is the smallest of the four; the fixed rows are exact zeros, so the atom carrying the largest force is always one that is free to move
    - Most of the free atoms are above the yardstick, not just the adsorbate: the surface has to move as well
- What module 07 does about it
    - It moves the free atoms along their forces, step by step, until the largest force falls below the chosen value, and the energy at the end is lower than the energy it started from
    - Every energy of this notebook therefore changes, and the changes are not the same for the four sites, so the ranking of the previous section can change too
    - The clean slab and the O2 reference have to be relaxed with the same settings, or the difference mixes a relaxed structure with an unrelaxed reference
- `magnitudes.argmax()` is the position of the largest value in an array, used here to find which atom carries the largest force
- `(magnitudes > 0.05).sum()` counts how many atoms are above the yardstick, fixed atoms included; their rows are zero, so they never count
- In the table:
    - One row per site, with the force magnitudes of the same single points the section on reusing a calculator ran
    - `hcp` carries the largest fmax while `ontop`, the highest of the four in energy, carries the smallest: a low energy does not mean a small force, and neither number substitutes for the other
    - 13 to 17 of the 19 free atoms are above the yardstick in every structure

In [6]:
rows = []
for site in SITES:
    atoms = structures[site]
    magnitudes = np.linalg.norm(forces_by_site[site], axis=1)
    i_worst = int(magnitudes.argmax())
    i_o = len(atoms) - 1
    rows.append({"site": site,
                 "fmax (eV/Å)": round(calc_fmax(forces_by_site[site]), 3),
                 "atom carrying it": atoms[i_worst].symbol,
                 "force on the O (eV/Å)": round(float(magnitudes[i_o]), 3),
                 "atoms above 0.05 eV/Å": int((magnitudes > 0.05).sum()),
                 "free atoms": len(atoms) - len(atoms.constraints[0].get_indices())})

print("Forces of the four single points, read against the 0.05 eV/Å yardstick")
print(pd.DataFrame(rows).to_string(index=False))

print()
print("Largest force on the clean slab, for comparison (eV/Å)")
print(round(calc_fmax(forces), 3))

Forces of the four single points, read against the 0.05 eV/Å yardstick
  site  fmax (eV/Å) atom carrying it  force on the O (eV/Å)  atoms above 0.05 eV/Å  free atoms
 ontop        0.995               Pt                  0.403                     13          19
bridge        1.514                O                  1.514                     17          19
   fcc        1.783                O                  1.783                     16          19
   hcp        2.023                O                  2.023                     17          19

Largest force on the clean slab, for comparison (eV/Å)
0.392


## 6. The same single point from the terminal: `mlip singlepoint run`

- `mliprun` 0.5.0 has a command that does what this notebook did from Python, for one structure in a file: `mlip singlepoint run`
    - Version 0.4.0, which earlier versions of this course were written against, had none; the cell prints the version of your environment first, and `mlip --help` lists `singlepoint` from 0.5.0 on
- A command reads its structure from a file, so the cell first writes the `fcc` structure of the section on reusing a calculator to `work/single_point_fcc/POSCAR`, the way module 05's section on saving the structures wrote its sites
    - `sort=True` groups the atoms by element, so the O is the first atom of the file and no longer the last
- `mlip singlepoint run --structure work/single_point_fcc/POSCAR --mlip mace --device cpu --no-stress`
    - `--mlip mace` names the model, canon `C1`, and `--device cpu` the processor, as the Python calls above did
    - `--no-stress` skips the stress, the force that acts on the cell itself, and for this structure it is needed rather than optional
        - The command computes a stress only for a cell that is periodic in all three directions, because along the vacuum of a slab a stress means nothing
        - A POSCAR file has no way to say that a direction is not periodic, so a slab read back from one is periodic in all three, and without the flag the command reports a stress for it that must not be read
    - `subprocess.run` starts a program the way the terminal would and waits for it to finish; `capture_output=True` keeps what it prints, and `result.returncode` is 0 when it succeeded, so the cell stops and shows the end of the error output when it did not
    - Module 07 wraps the same call in a small function, and uses it for every command from there on
- What the command writes, and what it adds over the Python cells above
    - `mliprun_run.json`, the run record: the model, the version of every package, the settings and the results, which is what makes a number traceable to the model that produced it; module 07's section on reading the outputs reads one in full
    - `singlepoint_forces.csv`, one row per atom with the raw force the model predicts on it, held atoms included, and three columns `free_x`, `free_y`, `free_z` saying which components are free
    - Two largest forces instead of one: `fmax_free_eV_per_A` over the free atoms, the number a relaxation converges against and the one `calc_fmax(atoms.get_forces())` gives, and `fmax_all_eV_per_A` with nothing held, the number section 2 got from a copy with the constraint removed
    - It builds the model on every run, as `setup_calculator` does, so for many structures of one study the Python of the section on reusing a calculator is the faster way
- The run record is replaced, not added to, by the next command that writes into the same folder
    - Run `mlip singlepoint run` on the `opt_final.vasp` of a relaxation, inside that relaxation's folder, and the relaxation's own record is gone, with no warning
    - `--output-dir <folder>` puts the single point's files in a folder of their own; this cell does not need it, because `work/single_point_fcc/` holds nothing else
- `pd.read_csv(file)` reads a CSV file into a table, as in module 01, and `table[~table["free_z"]]` keeps the rows whose `free_z` is `False`, the `~` turning every `True` into `False` and back
- In the printed output:
    - The version, the command as you would type it, and the seconds it took, most of them spent building the model
    - The stage of the record is of kind `singlepoint`, and `completed`
    - The energy agrees with the loop of the section on reusing a calculator to about 0.00002 eV, and not to the last digit, because the file lists the atoms in a different order: a model working in single precision, section 1's precision line, adds the same contributions up in a different order and the last digits move; put the atoms of the in-memory structure in the file's order and the model gives the command's number exactly, which was checked when this section was written
    - Both largest forces are 1.783 eV/Å, the `fcc` row of the section on forces; they agree because the atom carrying the largest force is the O, which is free, and on a structure where a held atom carried the largest force the two would differ
    - No stress, with the reason `not requested`, which is the `--no-stress` flag
    - The forces file has 28 rows, 9 of them for the held bottom layer, and the largest force on a held atom there is 0.398 eV/Å, where `atoms.get_forces()` reports exact zeros: the same point section 2 made on the clean slab, now written into a file
    - The folder holds the POSCAR the cell wrote and the two files the command wrote


In [7]:
from importlib.metadata import version

SP_DIR = Path("work/single_point_fcc")
SP_DIR.mkdir(parents=True, exist_ok=True)
write(SP_DIR / "POSCAR", pt111_with_adsorbate("O", "fcc"), format="vasp", direct=True, sort=True)

arguments = ["singlepoint", "run",
             "--structure", str(SP_DIR / "POSCAR"),
             "--mlip", "mace",
             "--device", "cpu",
             "--no-stress"]

print("mliprun version in this environment")
print(version("mliprun"))
print()
print("Command, as you would type it in the terminal")
print(" ".join(["mlip"] + arguments))

t_start = time.perf_counter()
result = subprocess.run([str(MLIP)] + arguments, capture_output=True, text=True)
print("Seconds the command took, the model build included")
print(round(time.perf_counter() - t_start, 1))
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise SystemExit("mlip command failed")

record = json.loads((SP_DIR / "mliprun_run.json").read_text())
stage = record["stages"][-1]
results = stage["results"]

print()
print("Kind and status of the stage in the run record")
print([stage["kind"], stage["status"]])
print("Energy from the command (eV)")
print(round(results["energy_eV"], 6))
print("Energy of the same structure from the loop of the section on reusing a calculator (eV)")
print(round(energies["fcc"], 6))
print("Largest force on a free atom, then on any atom with nothing held (eV/Å)")
print([round(results["fmax_free_eV_per_A"], 3), round(results["fmax_all_eV_per_A"], 3)])
print("Free atoms, and the symbol of the free atom carrying the largest force")
print([results["n_free_atoms"], results["worst_force_atom_free_symbol"]])
print("Stress, and the reason the record gives for it")
print([results["stress_GPa"], results["stress_unavailable_reason"]])
print("Model and mliprun version, as the record keeps them")
print([record["provenance"]["mlip_model"], record["provenance"]["mliprun_version"]])

table = pd.read_csv(SP_DIR / "singlepoint_forces.csv")
held = table[~table["free_z"]]

print()
print("Columns of singlepoint_forces.csv")
print(list(table.columns))
print("Rows in it, then rows of atoms held in place")
print([len(table), len(held)])
print("Largest force on a held atom in that file (eV/Å)")
print(round(float(held["f_norm"].max()), 3))
print("Files the command wrote")
print(sorted(path.name for path in SP_DIR.iterdir()))


mliprun version in this environment
0.5.0

Command, as you would type it in the terminal
mlip singlepoint run --structure work/single_point_fcc/POSCAR --mlip mace --device cpu --no-stress


Seconds the command took, the model build included
5.1

Kind and status of the stage in the run record
['singlepoint', 'completed']
Energy from the command (eV)
-158.197083
Energy of the same structure from the loop of the section on reusing a calculator (eV)
-158.197067
Largest force on a free atom, then on any atom with nothing held (eV/Å)
[1.783, 1.783]
Free atoms, and the symbol of the free atom carrying the largest force
[19, 'O']
Stress, and the reason the record gives for it
[None, 'not requested']
Model and mliprun version, as the record keeps them
['mace', '0.5.0']

Columns of singlepoint_forces.csv
['index', 'symbol', 'fx', 'fy', 'fz', 'f_norm', 'free_x', 'free_y', 'free_z']
Rows in it, then rows of atoms held in place
[28, 9]
Largest force on a held atom in that file (eV/Å)
0.398
Files the command wrote
['POSCAR', 'mliprun_run.json', 'singlepoint_forces.csv']


## Check yourself

1. You have `E(slab+O)` from a run with `mlip="mace"`, and you take `E(slab)` from a file left over from another
   project that used a UMA head. You subtract them and get an `E_ads` larger than any bond energy you know of.
   What went wrong, which canon item says so, and what do you have to rerun?
2. Your `fmax` for O on `fcc` is 1.783 eV/Å, and you write the `E_ads` of that structure into your notes as the
   adsorption energy of O on Pt(1 1 1). What is wrong with that, and what has to be run before any number from
   this notebook can be reported?
3. You loop over 40 structures and call `setup_calculator(atoms, mlip="mace", device="cpu")` inside the loop. It
   takes far longer than you expected, and a colleague's loop over the same 40 structures finishes in a fraction
   of the time. What is the difference between the two loops?
4. You relax a structure with `mlip optimize run` in `work/relax/`, then run
   `mlip singlepoint run --structure work/relax/opt_final.vasp --mlip mace` to check its forces. A week later you
   open `work/relax/mliprun_run.json` to report how many steps the relaxation took, and the record has no
   relaxation in it. What happened, and how do you run the single point so that it does not happen?

<details>
<summary>Answers</summary>

- The two terms come from two different models, whose energy zeros are independent: each was fitted separately,
  and the zero of a total energy is not part of what a fit constrains. The difference you computed is dominated
  by the gap between those two zeros, which is not a physical quantity. Canon `C3` is the rule, one head per
  energy formula, and canon `C1` is why the model and the head are pinned and written out in the first place.
  Rerun every term of the formula with one model and one head, the clean slab and the O2 reference included
- It is a single point on a geometry that was never relaxed: the O sits at a height taken from module 05's table
  of starting guesses. A force of 1.783 eV/Å is more than thirty times the 0.05 eV/Å that module 07 relaxes to,
  so the structure is nowhere near a minimum of this model, and the energy will fall once the atoms are allowed
  to move. It falls far enough to change the sign: relaxed with this same model and this same reference, the
  `E_ads` of that site comes out negative, so the positive number you were about to write down would have said
  that oxygen does not bind to platinum, which is wrong. The four sites will not fall by the same amount, so the
  ranking can change as well. Relax every term with the same settings, as module 07's section on adsorption
  energies from relaxed structures does, and quote the relaxed numbers
- `setup_calculator` builds a new model on every call, so that loop reads the weights from disk and into memory
  40 times, and the model build is the slow part: this notebook prints the seconds for one build and the seconds
  for four single points, and the build is larger by a wide margin. The colleague called
  `build_calculator(mlip="mace", device="cpu")` once before the loop and then assigned `atoms.calc = calc`
  inside it, which is the section on reusing one calculator
- The single point wrote its own `mliprun_run.json` into `work/relax/`, the folder of its input structure, and a run
  record is replaced, not added to, by the next command that writes into the same folder, so the relaxation's record
  is gone and nothing warned you. Give the single point a folder of its own with `--output-dir`, for example
  `--output-dir work/relax_check`; the relaxation's record then stays where it was

</details>